In [4]:
import pandas as pd

kelly = pd.read_csv('kelly_final.csv', nrows=1000)
print(kelly.columns.tolist())

# VALUE features in Kelly
VALUE = [
    'bm',       # book to market
    'bm_ia',    # industry-adjusted book to market
    'cfp',      # cashflow to price
    'cfp_ia',   # industry-adjusted cashflow to price
    'ep',       # earnings to price
    'sp',       # sales to price
    'dy',       # dividend yield
]

# QUALITY features in Kelly
QUALITY = [
    'roaq',     # ROA quarterly
    'roeq',     # ROE quarterly
    'gma',      # gross margin (= our gp_at)
    'operprof', # operating profitability
    'acc',      # accruals (lower = better quality)
    'absacc',   # absolute accruals
    'pctacc',   # percent accruals
    'stdacc',   # std of accruals
    'agr',      # asset growth
    'egr',      # equity growth
    'lgr',      # liability growth
    'lev',      # leverage
    'ms',       # Mohanram S-score
    'nincr',    # number of earnings increases
    'tb',       # tax to book
    'cashdebt', # cashflow to debt
]

# MOMENTUM features in Kelly
MOMENTUM = [
    'mom1m',    # 1-month
    'mom6m',    # 6-month
    'mom12m',   # 12-month
    'mom36m',   # 36-month
    'chmom',    # change in momentum
    'indmom',   # industry momentum
]

# Verify all exist in Kelly
all_features = VALUE + QUALITY + MOMENTUM
missing = [f for f in all_features if f not in kelly.columns]
present = [f for f in all_features if f in kelly.columns]

print(f"Features present in Kelly: {len(present)}/{len(all_features)}")
print(f"Missing: {missing}")

# Check coverage for each
print("\nValue feature coverage:")
for f in VALUE:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

print("\nQuality feature coverage:")
for f in QUALITY:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

print("\nMomentum feature coverage:")
for f in MOMENTUM:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

['permno', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd', 'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol', 'roeq', 'rsup', 'stdacc', 'stdcf', 'ms', 'baspread', 'ill', 'maxret', 'retvol', 'std_dolvol', 'std_turn', 'zerotrade', 'sic2', 'date', 'ret', 'ret_raw', 'me']
Features present in Kelly: 

In [5]:
# Load from middle of dataset to get representative coverage
kelly = pd.read_csv('kelly_final.csv', skiprows=range(1, 1000000), nrows=1000)
# skiprows skips the first 1M data rows, keeps header

print("Date range of sample:", kelly['date'].min(), "to", kelly['date'].max())

print("\nValue feature coverage:")
for f in VALUE:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

print("\nQuality feature coverage:")
for f in QUALITY:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

print("\nMomentum feature coverage:")
for f in MOMENTUM:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

Date range of sample: 1984-02-01 to 1984-02-01

Value feature coverage:
  bm: 75.7%
  bm_ia: 75.7%
  cfp: 68.0%
  cfp_ia: 68.0%
  ep: 76.2%
  sp: 76.0%
  dy: 76.2%

Quality feature coverage:
  roaq: 60.6%
  roeq: 60.6%
  gma: 73.5%
  operprof: 72.8%
  acc: 66.3%
  absacc: 66.3%
  pctacc: 66.3%
  stdacc: 38.2%
  agr: 73.6%
  egr: 73.6%
  lgr: 73.3%
  lev: 75.8%
  ms: 60.2%
  nincr: 60.9%
  tb: 67.3%
  cashdebt: 71.8%

Momentum feature coverage:
  mom1m: 99.4%
  mom6m: 93.0%
  mom12m: 85.6%
  mom36m: 76.5%
  chmom: 85.6%
  indmom: 100.0%


In [6]:
import wrds
conn = wrds.Connection(wrds_username='your_username')

# Get gvkeys from link table
link = pd.read_csv('link_table.csv', parse_dates=['linkdt','linkenddt'])
gvkeys = link['gvkey'].astype(str).str.strip().str.zfill(6).unique().tolist()
print(f"Pulling quarterly data for {len(gvkeys)} firms...")

# Pull quarterly Compustat
chunk_size = 500
chunks = [gvkeys[i:i+chunk_size] for i in range(0, len(gvkeys), chunk_size)]

all_chunks = []
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}/{len(chunks)}...")
    gvkey_str = ','.join([f"'{g}'" for g in chunk])
    
    q = conn.raw_sql(f"""
        SELECT
            gvkey,
            datadate,
            fyearq,
            fqtr,
            rdq,
            ibq,
            saleq,
            atq,
            ceqq,
            seqq,
            niq,
            oiadpq,
            oibdpq,
            dpq,
            actq,
            lctq,
            cheq,
            dlcq,
            dlttq,
            cshoq,
            prccq,
            xrdq
        FROM comp.fundq
        WHERE datadate BETWEEN '1950-01-01' AND '2024-12-31'
            AND gvkey IN ({gvkey_str})
            AND indfmt = 'INDL'
            AND datafmt = 'STD'
            AND popsrc = 'D'
            AND consol = 'C'
    """, date_cols=['datadate', 'rdq'])
    
    all_chunks.append(q)

compq = pd.concat(all_chunks, ignore_index=True)
compq = compq.sort_values(['gvkey','datadate'])
compq = compq.drop_duplicates(subset=['gvkey','datadate'], keep='last')

print(f"\nQuarterly rows: {len(compq):,}")
print(f"Date range: {compq['datadate'].min()} to {compq['datadate'].max()}")

compq.to_csv('compustat_quarterly.csv', index=False)
print("Saved: compustat_quarterly.csv")

conn.close()

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Pulling quarterly data for 6776 firms...
  Chunk 1/14...
  Chunk 2/14...
  Chunk 3/14...
  Chunk 4/14...
  Chunk 5/14...
  Chunk 6/14...
  Chunk 7/14...
  Chunk 8/14...
  Chunk 9/14...
  Chunk 10/14...
  Chunk 11/14...
  Chunk 12/14...
  Chunk 13/14...
  Chunk 14/14...

Quarterly rows: 518,703
Date range: 1961-04-30 00:00:00 to 2024-12-31 00:00:00
Saved: compustat_quarterly.csv


In [3]:
# Add txt to the compustat pull
# Re-run just this part

import wrds
import pandas as pd

conn = wrds.Connection(wrds_username='muhab')

link = pd.read_csv('link_table.csv')
link['gvkey'] = link['gvkey'].astype(str).str.strip().str.zfill(6)
gvkeys = link['gvkey'].unique().tolist()

chunk_size = 500
chunks = [gvkeys[i:i+chunk_size] for i in range(0, len(gvkeys), chunk_size)]

all_chunks = []
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}/{len(chunks)}...")
    gvkey_str = ','.join([f"'{g}'" for g in chunk])
    
    q = conn.raw_sql(f"""
        SELECT
            gvkey, datadate, fyear, fyr,
            at, lt, seq, ceq,
            pstk, pstkrv, pstkl,
            txditc, txp, txt,
            act, lct, che, rect, invt,
            dlc, dltt,
            ni, ib, oiadp, oibdp,
            dp, sale, cogs, xsga, xrd,
            capx, dv, csho, prcc_f,
            naicsh, sich
        FROM comp.funda
        WHERE datadate BETWEEN '1950-01-01' AND '2024-12-31'
            AND gvkey IN ({gvkey_str})
            AND indfmt = 'INDL'
            AND datafmt = 'STD'
            AND popsrc = 'D'
            AND consol = 'C'
            AND at > 0
    """, date_cols=['datadate'])
    
    all_chunks.append(q)

comp = pd.concat(all_chunks, ignore_index=True)
comp = comp.sort_values(['gvkey','datadate'])
comp = comp.drop_duplicates(subset=['gvkey','datadate'], keep='last')

print(f"Rows: {len(comp):,}")
print(f"txt coverage: {comp['txt'].notna().mean():.1%}")

comp.to_csv('compustat_annual.csv', index=False)
print("Saved: compustat_annual.csv")
conn.close()

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
  Chunk 1/14...
  Chunk 2/14...
  Chunk 3/14...
  Chunk 4/14...
  Chunk 5/14...
  Chunk 6/14...
  Chunk 7/14...
  Chunk 8/14...
  Chunk 9/14...
  Chunk 10/14...
  Chunk 11/14...
  Chunk 12/14...
  Chunk 13/14...
  Chunk 14/14...
Rows: 123,085
txt coverage: 99.6%
Saved: compustat_annual.csv


In [5]:
import pandas as pd
import numpy as np
import gc

print("Loading files...")
crsp  = pd.read_csv('crsp_returns.csv', parse_dates=['date'])
comp  = pd.read_csv('compustat_annual.csv', parse_dates=['datadate'])
compq = pd.read_csv('compustat_quarterly.csv', 
                     parse_dates=['datadate','rdq'])
link  = pd.read_csv('link_table.csv', 
                     parse_dates=['linkdt','linkenddt'])

for df in [comp, compq, link]:
    df['gvkey'] = df['gvkey'].astype(str).str.strip().str.zfill(6)
link['permno'] = link['permno'].astype(int)

# ============================================================
# STEP 1: PREPARE CRSP (2017+ only)
# ============================================================

ext_permnos = crsp[crsp['date'] >= '2022-01-01']['permno'].unique()
crsp_ext = crsp[
    (crsp['permno'].isin(ext_permnos)) &
    (crsp['date'] >= '2017-01-01')
].copy()

crsp_ext = crsp_ext.merge(
    link[['permno','gvkey','linkdt','linkenddt']],
    on='permno', how='left'
)
crsp_ext = crsp_ext[
    (crsp_ext['date'] >= crsp_ext['linkdt']) &
    (crsp_ext['date'] <= crsp_ext['linkenddt'])
].copy()
crsp_ext = crsp_ext.dropna(subset=['gvkey'])
crsp_ext['gvkey'] = crsp_ext['gvkey'].astype(str).str.strip().str.zfill(6)

# Free link table
del link
gc.collect()

print(f"CRSP ext rows: {len(crsp_ext):,}")
ext_gvkeys = set(crsp_ext['gvkey'].unique())

# ============================================================
# STEP 2: PRE-FILTER AND PREPARE ANNUAL COMPUSTAT
# Only keep gvkeys we need AND years that could be relevant
# For 2020-2024 CRSP data, we need accounting from 2014+
# (6 month lag means 2015 accounting available mid-2015)
# ============================================================

comp_clean = comp[
    (comp['gvkey'].isin(ext_gvkeys)) &
    (comp['datadate'] >= '2014-01-01')  # only recent years needed
].copy()

print(f"Filtered annual comp: {len(comp_clean):,} rows")

comp_clean['available_date'] = (
    (comp_clean['datadate'] + pd.DateOffset(months=6))
    .dt.to_period('M').dt.to_timestamp()
)
comp_clean = comp_clean.sort_values(['gvkey','available_date'])
comp_clean['next_available'] = (
    comp_clean.groupby('gvkey')['available_date']
    .shift(-1).fillna(pd.Timestamp('2099-12-31'))
)

del comp
gc.collect()

# ============================================================
# STEP 3: PRE-FILTER AND PREPARE QUARTERLY COMPUSTAT
# Same logic — only recent years and relevant gvkeys
# ============================================================

compq_clean = compq[
    (compq['gvkey'].isin(ext_gvkeys)) &
    (compq['datadate'] >= '2018-01-01')  # need 16 quarters for stdacc
].copy()

print(f"Filtered quarterly comp: {len(compq_clean):,} rows")

compq_clean['available_date'] = (
    compq_clean['rdq']
    .fillna(compq_clean['datadate'] + pd.DateOffset(months=3))
    .dt.to_period('M').dt.to_timestamp()
)
compq_clean = compq_clean.sort_values(['gvkey','available_date'])
compq_clean['next_available_q'] = (
    compq_clean.groupby('gvkey')['available_date']
    .shift(-1).fillna(pd.Timestamp('2099-12-31'))
)

del compq
gc.collect()

# ============================================================
# STEP 4: EXPAND QUARTERLY TO MONTHLY PANEL
# Do this BEFORE merging with CRSP
# ============================================================

print("Expanding quarterly to monthly panel...")

date_range = pd.date_range(start='2017-01-01', end='2024-11-01', freq='MS')

q_cols = ['ibq','atq','ceqq','niq','oiadpq','dpq',
          'actq','lctq','cheq','dlcq','dlttq','xrdq']

monthly_q_list = []

for gvkey, firm_data in compq_clean.groupby('gvkey'):
    firm_rows = []
    for _, row in firm_data.iterrows():
        valid_months = date_range[
            (date_range >= row['available_date']) &
            (date_range <  row['next_available_q'])
        ]
        if len(valid_months) == 0:
            continue
        for month in valid_months:
            firm_rows.append({
                'gvkey': gvkey,
                'date':  month,
                **{col: row[col] for col in q_cols if col in row.index}
            })
    if firm_rows:
        monthly_q_list.append(pd.DataFrame(firm_rows))

monthly_q = pd.concat(monthly_q_list, ignore_index=True)
monthly_q = monthly_q.drop_duplicates(subset=['gvkey','date'], keep='last')

print(f"Monthly quarterly panel: {len(monthly_q):,} rows")

del compq_clean, monthly_q_list
gc.collect()

# ============================================================
# STEP 5: MERGE ANNUAL — now safe, both sides are small
# ============================================================

print("Merging annual Compustat...")

# Process in chunks by gvkey to stay within memory
ann_chunks = []

for gvkey, crsp_firm in crsp_ext.groupby('gvkey'):
    comp_firm = comp_clean[comp_clean['gvkey'] == gvkey]
    if len(comp_firm) == 0:
        ann_chunks.append(crsp_firm)
        continue
    
    merged_firm = crsp_firm.merge(comp_firm, on='gvkey', how='left')
    merged_firm = merged_firm[
        (merged_firm['date'] >= merged_firm['available_date']) &
        (merged_firm['date'] <  merged_firm['next_available'])
    ]
    merged_firm = merged_firm.drop_duplicates(subset=['permno','date'], 
                                               keep='last')
    ann_chunks.append(merged_firm)

merged = pd.concat(ann_chunks, ignore_index=True)

del ann_chunks, comp_clean
gc.collect()

print(f"After annual merge: {len(merged):,}")
print(f"Annual match rate: {merged['at'].notna().mean():.2%}")

# ============================================================
# STEP 6: MERGE QUARTERLY — simple join, no explosion
# ============================================================

print("Merging quarterly...")
merged = merged.merge(monthly_q, on=['gvkey','date'], how='left')

del monthly_q
gc.collect()

print(f"After quarterly merge: {len(merged):,}")
print(f"Quarterly match rate: {merged['atq'].notna().mean():.2%}")

# ============================================================
# STEP 7: COMPUTE FEATURES
# ============================================================

def safe_div(a, b):
    a = pd.to_numeric(a, errors='coerce')
    b = pd.to_numeric(b, errors='coerce')
    return np.where((b != 0) & b.notna() & a.notna(), a/b, np.nan)

def compute_all_features(df):
    out = df.copy()
    out = out.sort_values(['permno','date']).reset_index(drop=True)

    # Book equity
    out['be'] = (
        out['seq']
        .fillna(out['ceq'] + out['pstk'].fillna(0))
        .fillna(out['at'] - out['lt'])
    )
    out['be'] = (out['be']
                 - out['pstkrv']
                   .fillna(out['pstkl'])
                   .fillna(out['pstk'])
                   .fillna(0))
    out['be'] = out['be'] + out['txditc'].fillna(0)
    out['be'] = out['be'].clip(lower=0.001)

    out['sic2'] = out['siccd'].astype(str).str[:2]

    # VALUE
    out['bm']  = safe_div(out['be'], out['me'])
    out['ep']  = safe_div(out['ib'], out['me'])
    out['cfp'] = safe_div(out['ib'].fillna(0) + out['dp'].fillna(0), out['me'])
    out['sp']  = safe_div(out['sale'], out['me'])
    out['dy']  = safe_div(out['dv'], out['me'])
    ind_med_bm  = out.groupby(['date','sic2'])['bm'].transform('median')
    ind_med_cfp = out.groupby(['date','sic2'])['cfp'].transform('median')
    out['bm_ia']  = out['bm']  - ind_med_bm
    out['cfp_ia'] = out['cfp'] - ind_med_cfp

    # QUALITY — profitability
    out['roaq']     = safe_div(out['ibq'], out['atq'])
    out['roeq']     = safe_div(out['ibq'], out['ceqq'])
    out['gma']      = safe_div(out['sale'] - out['cogs'].fillna(0), out['at'])
    out['operprof'] = safe_div(out['oiadp'] - out['xsga'].fillna(0), out['be'])

    # QUALITY — accruals
    out['act_lag']  = out.groupby('permno')['act'].shift(1)
    out['lct_lag']  = out.groupby('permno')['lct'].shift(1)
    out['at_lag1']  = out.groupby('permno')['at'].shift(1)
    out['avg_at']   = (out['at'] + out['at_lag1']) / 2
    out['delta_wc'] = (
        (out['act'] - out['che'].fillna(0)) -
        (out['lct'] - out['dlc'].fillna(0))
    ) - (
        (out['act_lag'] - out['che'].fillna(0)) -
        (out['lct_lag'] - out['dlc'].fillna(0))
    )
    out['acc']    = safe_div(out['delta_wc'] - out['dp'].fillna(0), out['avg_at'])
    out['absacc'] = out['acc'].abs()
    out['pctacc'] = safe_div(out['acc'], out['ni'].abs())
    out['txp_lag'] = out.groupby('permno')['txp'].shift(1)
    out['tb']      = safe_div(
        out['txt'].fillna(0) - (out['txp'].fillna(0) - out['txp_lag'].fillna(0)),
        out['ib'].abs()
    )
    out['cashdebt'] = safe_div(
        out['ib'].fillna(0) + out['dp'].fillna(0),
        out['dltt'].fillna(0) + out['dlc'].fillna(0)
    )
    out['acc_q']  = safe_div(out['ibq'].fillna(0) - out['oiadpq'].fillna(0), out['atq'])
    out['stdacc'] = out.groupby('permno')['acc_q'].transform(
        lambda x: x.rolling(16, min_periods=8).std()
    )

    # QUALITY — growth
    out['agr']     = safe_div(out['at'] - out['at_lag1'], out['at_lag1'])
    out['ceq_lag'] = out.groupby('permno')['ceq'].shift(1)
    out['egr']     = safe_div(out['ceq'] - out['ceq_lag'], out['ceq_lag'])
    out['lt_lag']  = out.groupby('permno')['lt'].shift(1)
    out['lgr']     = safe_div(out['lt'] - out['lt_lag'], out['lt_lag'])
    out['ppegt']     = out['at'] - out['act'].fillna(0) - out['che'].fillna(0)
    out['ppegt_lag'] = out.groupby('permno')['ppegt'].shift(1)
    out['invt_lag']  = out.groupby('permno')['invt'].shift(1)
    out['invest']    = safe_div(
        (out['ppegt'] - out['ppegt_lag'].fillna(0)) +
        (out['invt'].fillna(0) - out['invt_lag'].fillna(0)),
        out['at_lag1']
    )

    # QUALITY — financial health
    out['debt'] = out['dltt'].fillna(0) + out['dlc'].fillna(0)
    out['lev']  = safe_div(out['debt'], out['at'])

    out['cfo_at']     = safe_div(out['oiadpq'], out['atq'])
    ind_med_roa       = out.groupby(['date','sic2'])['roaq'].transform('median')
    ind_med_cfo       = out.groupby(['date','sic2'])['cfo_at'].transform('median')
    out['xrd_lag']    = out.groupby('permno')['xrd'].shift(1)
    out['capx_lag']   = out.groupby('permno')['capx'].shift(1)
    out['ato']        = safe_div(out['sale'], out['at'])
    out['ato_lag']    = out.groupby('permno')['ato'].shift(1)
    out['margin']     = safe_div(out['ib'], out['sale'])
    out['margin_lag'] = out.groupby('permno')['margin'].shift(1)
    out['csho_lag']   = out.groupby('permno')['csho'].shift(1)

    s1 = (out['roaq']   > ind_med_roa).astype(float)
    s2 = (out['cfo_at'] > ind_med_cfo).astype(float)
    s3 = (out['oiadpq'].fillna(0) > out['ibq'].fillna(0)).astype(float)
    s4 = (out['xrd'].fillna(0)    > out['xrd_lag'].fillna(0)).astype(float)
    s5 = (out['capx'].fillna(0)   > out['capx_lag'].fillna(0)).astype(float)
    s6 = (out['ato']    > out['ato_lag']).astype(float)
    s7 = (out['margin'] > out['margin_lag']).astype(float)
    s8 = (out['csho']   <= out['csho_lag']).astype(float)
    out['ms'] = s1+s2+s3+s4+s5+s6+s7+s8

    out['ib_q_lag'] = out.groupby('permno')['ibq'].shift(1)
    out['earn_up']  = (out['ibq'] > out['ib_q_lag']).astype(float)
    out['nincr']    = out.groupby('permno')['earn_up'].transform(
        lambda x: x.rolling(8, min_periods=1).sum()
    )

    # MOMENTUM
    out['mom1m']  = out.groupby('permno')['ret'].shift(1)
    out['mom6m']  = out.groupby('permno')['ret'].transform(
        lambda x: x.shift(2).rolling(5)
        .apply(lambda r: (1+r).prod()-1, raw=True)
    )
    out['mom12m'] = out.groupby('permno')['ret'].transform(
        lambda x: x.shift(2).rolling(11)
        .apply(lambda r: (1+r).prod()-1, raw=True)
    )
    out['mom36m'] = out.groupby('permno')['ret'].transform(
        lambda x: x.shift(13).rolling(24)
        .apply(lambda r: (1+r).prod()-1, raw=True)
    )
    out['mom12m_lag6'] = out.groupby('permno')['mom12m'].shift(6)
    out['chmom']  = out['mom12m'] - out['mom12m_lag6']
    out['indmom'] = out.groupby(['date','sic2'])['mom12m'].transform('mean')

    return out

print("\nComputing features...")
panel = compute_all_features(merged)

del merged
gc.collect()

# ============================================================
# STEP 8: FILTER AND SAVE
# ============================================================

VALUE    = ['bm','bm_ia','cfp','cfp_ia','ep','sp','dy']
QUALITY  = ['roaq','roeq','gma','operprof','acc','absacc',
            'pctacc','tb','cashdebt','stdacc','agr','egr',
            'lgr','invest','lev','ms','nincr']
MOMENTUM = ['mom1m','mom6m','mom12m','mom36m','chmom','indmom']
ALL_FEATURES = VALUE + QUALITY + MOMENTUM

panel_ext = panel[panel['date'] >= '2022-01-01'].copy()

output_cols = ['permno','date','ret','me','sic2'] + ALL_FEATURES
output_cols = [c for c in output_cols if c in panel_ext.columns]
panel_ext   = panel_ext[output_cols]

print(f"\nShape: {panel_ext.shape}")
print(f"Date range: {panel_ext['date'].min()} to {panel_ext['date'].max()}")
print(f"Unique permnos: {panel_ext['permno'].nunique():,}")

print("\nFeature coverage:")
for f in ALL_FEATURES:
    if f in panel_ext.columns:
        cov = panel_ext[f].notna().mean()
        flag = " <-- LOW" if cov < 0.6 else ""
        print(f"  {f}: {cov:.1%}{flag}")

panel_ext.to_csv('features_2022_2024.csv', index=False)
print("\nSaved: features_2022_2024.csv")

Loading files...
CRSP ext rows: 457,886
Filtered annual comp: 51,304 rows
Filtered quarterly comp: 153,591 rows
Expanding quarterly to monthly panel...
Monthly quarterly panel: 461,019 rows
Merging annual Compustat...
After annual merge: 449,769
Annual match rate: 89.27%
Merging quarterly...
After quarterly merge: 449,769
Quarterly match rate: 75.69%

Computing features...

Shape: (180123, 35)
Date range: 2022-01-01 00:00:00 to 2024-11-01 00:00:00
Unique permnos: 6,282

Feature coverage:
  bm: 88.7%
  bm_ia: 88.7%
  cfp: 100.0%
  cfp_ia: 100.0%
  ep: 88.6%
  sp: 88.6%
  dy: 87.7%
  roaq: 86.9%
  roeq: 86.7%
  gma: 88.6%
  operprof: 88.6%
  acc: 71.5%
  absacc: 71.5%
  pctacc: 71.4%
  tb: 88.6%
  cashdebt: 85.6%
  stdacc: 85.8%
  agr: 88.6%
  egr: 88.4%
  lgr: 88.4%
  invest: 88.6%
  lev: 88.7%
  ms: 100.0%
  nincr: 100.0%
  mom1m: 99.9%
  mom6m: 98.8%
  mom12m: 95.8%
  mom36m: 80.5%
  chmom: 91.7%
  indmom: 100.0%

Saved: features_2022_2024.csv


In [6]:
VALUE    = ['bm','bm_ia','cfp','cfp_ia','ep','sp','dy']

QUALITY  = ['roaq','roeq','gma','operprof',
            'acc','absacc','pctacc',
            'tb','cashdebt','stdacc',
            'agr','egr','lgr','invest',
            'lev','ms','nincr']

MOMENTUM = ['mom1m','mom6m','mom12m','mom36m','chmom','indmom']

VALUE_ONLY         = VALUE                        # Model A: 7 features
VALUE_QUALITY      = VALUE + QUALITY              # Model B: 24 features
VALUE_QUALITY_MOM  = VALUE + QUALITY + MOMENTUM   # Model C: 30 features

print(f"Model A — Value only:              {len(VALUE_ONLY)} features")
print(f"Model B — Value + Quality:         {len(VALUE_QUALITY)} features")
print(f"Model C — Value + Quality + Mom:   {len(VALUE_QUALITY_MOM)} features")

# Final verification both datasets have all features
import pandas as pd

kelly = pd.read_csv('kelly_final.csv', nrows=5)
ext   = pd.read_csv('features_2022_2024.csv', nrows=5)

for name, cols in [('VALUE', VALUE), 
                    ('QUALITY', QUALITY), 
                    ('MOMENTUM', MOMENTUM)]:
    missing_k = [f for f in cols if f not in kelly.columns]
    missing_e = [f for f in cols if f not in ext.columns]
    print(f"\n{name}:")
    print(f"  Missing from Kelly:     {missing_k if missing_k else 'none'}")
    print(f"  Missing from extension: {missing_e if missing_e else 'none'}")

Model A — Value only:              7 features
Model B — Value + Quality:         24 features
Model C — Value + Quality + Mom:   30 features

VALUE:
  Missing from Kelly:     none
  Missing from extension: none

QUALITY:
  Missing from Kelly:     none
  Missing from extension: none

MOMENTUM:
  Missing from Kelly:     none
  Missing from extension: none
